# End-to-End Image Classification with Transfer Learning

---

## Learning Objectives

By the end of this project, you will:

- Build a complete, production-quality image classification pipeline from scratch
- Create proper train/validation/test splits with augmentation on training data only
- Train a baseline CNN from scratch and compare against transfer learning approaches
- Fine-tune a pretrained ResNet18 with frozen and unfrozen layers
- Evaluate models with accuracy, confusion matrices, and per-class metrics
- Perform error analysis on misclassified examples
- Save and load models for inference

## Prerequisites

- **DL100**: Neural network fundamentals (forward/backward pass, activations)
- **DL150**: PyTorch foundations (tensors, `nn.Module`, optimizers, DataLoaders)
- **DL200**: MLP practical (training loops, evaluation)
- **DL300**: CNN fundamentals (convolution, pooling, feature maps)
- Basic understanding of transfer learning concepts

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Problem Definition and Dataset](#2.-Problem-Definition-and-Dataset)
3. [Data Exploration](#3.-Data-Exploration)
4. [Data Pipeline: Splits, Augmentation, DataLoaders](#4.-Data-Pipeline)
5. [Baseline Model: Simple CNN from Scratch](#5.-Baseline-Model:-Simple-CNN)
6. [Transfer Learning: Pretrained ResNet18](#6.-Transfer-Learning:-Pretrained-ResNet18)
7. [Fine-Tuning: Unfreeze and Lower LR](#7.-Fine-Tuning)
8. [Evaluation: Accuracy, Confusion Matrix, Per-Class Metrics](#8.-Evaluation)
9. [Error Analysis: Misclassified Examples](#9.-Error-Analysis)
10. [Model Saving and Inference](#10.-Model-Saving-and-Inference)
11. [Results Comparison](#11.-Results-Comparison)
12. [Conclusions and Next Steps](#12.-Conclusions-and-Next-Steps)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "..")
from src.utils.seed import set_global_seed
from src.utils.device import get_device
from src.utils.plotting import plot_loss_curves, plot_confusion_matrix
from src.utils.metrics import compute_accuracy, compute_classification_report
from src.utils.training import EarlyStopping, fit
from src.utils.data_helpers import safe_download_dataset

import numpy as np
import matplotlib.pyplot as plt
import time
import os
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix

set_global_seed(42)
device = get_device()

# Project paths
SAVE_DIR = os.path.join("..", "data", "models")
os.makedirs(SAVE_DIR, exist_ok=True)

print("Setup complete.")

---

## 2. Problem Definition and Dataset

**Task:** Multi-class image classification on CIFAR-10.

- **10 classes:** airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
- **50,000** training images, **10,000** test images
- **32x32** RGB images

**Approach:** We will compare three strategies:

| Approach | Description |
|----------|-------------|
| Baseline CNN | Simple CNN trained from scratch on CIFAR-10 |
| Transfer Learning | Pretrained ResNet18 with frozen backbone, only train new head |
| Fine-Tuning | Unfreeze last ResNet layers, train end-to-end with lower LR |

If CIFAR-10 download fails, we fall back to a synthetic image dataset.

In [ ]:
set_global_seed(42)

# CIFAR-10 class names
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
NUM_CLASSES = len(CLASS_NAMES)

# Try downloading CIFAR-10
DATA_DIR = os.path.join("..", "data", "raw")
os.makedirs(DATA_DIR, exist_ok=True)

# Basic transforms for initial loading (no augmentation yet)
basic_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_full = safe_download_dataset(
    datasets.CIFAR10, root=DATA_DIR, train=True, transform=basic_transform
)
test_dataset_raw = safe_download_dataset(
    datasets.CIFAR10, root=DATA_DIR, train=False, transform=basic_transform
)

USE_CIFAR = train_full is not None and test_dataset_raw is not None

if USE_CIFAR:
    print(f"CIFAR-10 loaded successfully.")
    print(f"  Training samples: {len(train_full)}")
    print(f"  Test samples:     {len(test_dataset_raw)}")
    print(f"  Image shape:      {train_full[0][0].shape}")
    print(f"  Classes:          {NUM_CLASSES}")
else:
    print("CIFAR-10 unavailable. Creating synthetic image dataset...")

In [ ]:
# Fallback: create synthetic image dataset if CIFAR-10 is unavailable
if not USE_CIFAR:
    from sklearn.datasets import make_classification

    set_global_seed(42)
    NUM_CLASSES = 10
    CLASS_NAMES = [f"class_{i}" for i in range(NUM_CLASSES)]
    IMG_SIZE = 32

    # Create synthetic "images": random patterns with class-dependent structure
    n_train, n_test = 5000, 1000
    n_total = n_train + n_test

    # Generate features and reshape to images
    X_flat, y_all = make_classification(
        n_samples=n_total, n_features=IMG_SIZE * IMG_SIZE * 3,
        n_informative=200, n_classes=NUM_CLASSES,
        n_clusters_per_class=1, random_state=42
    )
    # Normalize to [0, 1] and reshape to images
    X_flat = (X_flat - X_flat.min()) / (X_flat.max() - X_flat.min())
    X_images = X_flat.reshape(-1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)

    X_train_np, X_test_np = X_images[:n_train], X_images[n_train:]
    y_train_np, y_test_np = y_all[:n_train], y_all[n_train:]

    train_full = TensorDataset(
        torch.tensor(X_train_np), torch.tensor(y_train_np, dtype=torch.long)
    )
    test_dataset_raw = TensorDataset(
        torch.tensor(X_test_np), torch.tensor(y_test_np, dtype=torch.long)
    )

    print(f"Synthetic dataset created.")
    print(f"  Training samples: {len(train_full)}")
    print(f"  Test samples:     {len(test_dataset_raw)}")
    print(f"  Image shape:      {train_full[0][0].shape}")

---

## 3. Data Exploration

Before training, we visualize:
- Sample images from each class
- Class distribution to check for imbalance

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

# Gather one example per class
shown_classes = set()
for img, label in train_full:
    if isinstance(label, torch.Tensor):
        label = label.item()
    if label not in shown_classes:
        ax = axes[label]
        if isinstance(img, torch.Tensor):
            img_np = img.numpy()
        else:
            img_np = np.array(img)
        # Channel-first to channel-last for display
        if img_np.shape[0] == 3:
            img_np = np.transpose(img_np, (1, 2, 0))
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(CLASS_NAMES[label], fontsize=11)
        ax.axis("off")
        shown_classes.add(label)
    if len(shown_classes) == NUM_CLASSES:
        break

plt.suptitle("Sample Images from Each Class", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
all_labels = []
for _, label in train_full:
    if isinstance(label, torch.Tensor):
        all_labels.append(label.item())
    else:
        all_labels.append(label)

all_labels = np.array(all_labels)
class_counts = np.bincount(all_labels, minlength=NUM_CLASSES)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASS_NAMES, class_counts, color=plt.cm.tab10(np.arange(NUM_CLASSES)))
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title("Training Set Class Distribution")
plt.xticks(rotation=45, ha="right")

# Annotate bars
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha="center", fontsize=9)

ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Class distribution: min={class_counts.min()}, max={class_counts.max()}, "
      f"ratio={class_counts.max()/class_counts.min():.2f}")

---

## 4. Data Pipeline

Key design decisions:
- **Train/Val/Test split:** 80% train, 10% validation, 10% test (CIFAR already has a test set)
- **Augmentation on training data only:** random crop, horizontal flip, color jitter
- **Normalization:** ImageNet statistics (for transfer learning compatibility)
- **Resize to 32x32** for CNN baseline, **224x224** for ResNet

In [ ]:
set_global_seed(42)

# Split training into train + validation
n_total = len(train_full)
n_val = int(0.1 * n_total)
n_train = n_total - n_val

train_subset, val_subset = random_split(
    train_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_dataset_raw)}")

In [ ]:
# --- Transforms for baseline CNN (32x32) ---
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform_32 = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

eval_transform_32 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# --- Transforms for ResNet (224x224) ---
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform_224 = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform_224 = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transforms defined.")

In [ ]:
class TransformDataset(torch.utils.data.Dataset):
    """Wraps a dataset/subset and applies a transform to images.

    If the source images are already tensors (e.g., synthetic fallback),
    only normalization-style transforms are applied.
    """

    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform is not None:
            # If already a tensor and transform expects PIL, convert
            if isinstance(img, torch.Tensor):
                from torchvision.transforms.functional import to_pil_image
                img = to_pil_image(img)
            img = self.transform(img)
        return img, label


def build_loaders(train_sub, val_sub, test_ds, train_tfm, eval_tfm, batch_size=128):
    """Build DataLoaders with appropriate transforms."""
    train_ds = TransformDataset(train_sub, train_tfm)
    val_ds = TransformDataset(val_sub, eval_tfm)
    test_ds_t = TransformDataset(test_ds, eval_tfm)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds_t, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True)
    return train_loader, val_loader, test_loader


# Build 32x32 loaders for baseline CNN
train_loader_32, val_loader_32, test_loader_32 = build_loaders(
    train_subset, val_subset, test_dataset_raw,
    train_transform_32, eval_transform_32, batch_size=128
)

# Verify shapes
imgs, labels = next(iter(train_loader_32))
print(f"32x32 batch shape: {imgs.shape}, labels shape: {labels.shape}")

In [ ]:
# Visualize augmented vs original
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

# Get 5 raw images
raw_imgs = []
for i in range(5):
    img, _ = train_subset[i]
    if isinstance(img, torch.Tensor):
        raw_imgs.append(img.numpy().transpose(1, 2, 0))
    else:
        raw_imgs.append(np.array(img) / 255.0)

# Get 5 augmented images
aug_ds = TransformDataset(train_subset, train_transform_32)
aug_imgs = []
for i in range(5):
    img, _ = aug_ds[i]
    # Denormalize for display
    img_np = img.numpy().transpose(1, 2, 0)
    img_np = img_np * np.array(CIFAR_STD) + np.array(CIFAR_MEAN)
    aug_imgs.append(np.clip(img_np, 0, 1))

for i in range(5):
    axes[0, i].imshow(np.clip(raw_imgs[i], 0, 1))
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")

    axes[1, i].imshow(aug_imgs[i])
    axes[1, i].set_title("Augmented")
    axes[1, i].axis("off")

plt.suptitle("Data Augmentation: Original vs Augmented", fontsize=14)
plt.tight_layout()
plt.show()

---

## 5. Baseline Model: Simple CNN

Architecture:

```
Conv2d(3, 32, 3) -> BN -> ReLU -> MaxPool
Conv2d(32, 64, 3) -> BN -> ReLU -> MaxPool
Conv2d(64, 128, 3) -> BN -> ReLU -> AdaptiveAvgPool
Flatten -> Linear(128, 256) -> ReLU -> Dropout -> Linear(256, 10)
```

In [ ]:
class BaselineCNN(nn.Module):
    """Simple CNN for 32x32 image classification."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 32x32 -> 16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 16x16 -> 8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 8x8 -> 1x1 (adaptive)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


set_global_seed(42)
baseline_cnn = BaselineCNN(NUM_CLASSES).to(device)
print(baseline_cnn)

n_params = sum(p.numel() for p in baseline_cnn.parameters())
print(f"\nTotal parameters: {n_params:,}")

In [ ]:
# Training helper that tracks accuracy alongside loss
def train_classifier(model, train_loader, val_loader, epochs=20,
                     lr=1e-3, weight_decay=1e-4, patience=5, device=device):
    """Train a classifier with early stopping. Returns history dict."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, verbose=False
    )
    loss_fn = nn.CrossEntropyLoss()
    early_stop = EarlyStopping(patience=patience)

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": []
    }

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            correct += (output.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # --- Validate ---
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = loss_fn(output, y_batch)
                running_loss += loss.item() * X_batch.size(0)
                correct += (output.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        scheduler.step(val_loss)

        if epoch % 5 == 0 or epoch == 1:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
                  f"LR: {current_lr:.1e}")

        if early_stop(val_loss, model):
            print(f"Early stopping at epoch {epoch}")
            early_stop.load_best(model)
            break

    elapsed = time.time() - start_time
    history["training_time"] = elapsed
    print(f"\nTraining completed in {elapsed:.1f}s")
    return history

In [ ]:
# Train baseline CNN
set_global_seed(42)
baseline_cnn = BaselineCNN(NUM_CLASSES).to(device)

print("=" * 60)
print("Training Baseline CNN (from scratch)")
print("=" * 60)

history_baseline = train_classifier(
    baseline_cnn, train_loader_32, val_loader_32,
    epochs=30, lr=1e-3, patience=7
)

In [ ]:
# Plot baseline training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history_baseline["train_loss"]) + 1)
ax1.plot(epochs, history_baseline["train_loss"], label="Train Loss", marker="o", markersize=3)
ax1.plot(epochs, history_baseline["val_loss"], label="Val Loss", marker="s", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Baseline CNN: Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history_baseline["train_acc"], label="Train Acc", marker="o", markersize=3)
ax2.plot(epochs, history_baseline["val_acc"], label="Val Acc", marker="s", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Baseline CNN: Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Transfer Learning: Pretrained ResNet18

**Strategy:** Load a ResNet18 pretrained on ImageNet, freeze the backbone, replace the
classification head with a new one for our 10 classes.

Key considerations:
- ResNet18 expects 224x224 images with ImageNet normalization
- Only the new head parameters are trained
- This is fast since most parameters are frozen

In [ ]:
# Build 224x224 loaders for ResNet (smaller batch size due to larger images)
train_loader_224, val_loader_224, test_loader_224 = build_loaders(
    train_subset, val_subset, test_dataset_raw,
    train_transform_224, eval_transform_224, batch_size=64
)

imgs, labels = next(iter(train_loader_224))
print(f"224x224 batch shape: {imgs.shape}, labels shape: {labels.shape}")

In [ ]:
def create_resnet18_transfer(num_classes, freeze_backbone=True):
    """Create a ResNet18 model with a new classification head.

    Args:
        num_classes: Number of output classes.
        freeze_backbone: If True, freeze all backbone parameters.

    Returns:
        model: Modified ResNet18.
    """
    try:
        # Try new-style weights API first
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    except Exception:
        model = models.resnet18(pretrained=True)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the final fully-connected layer
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )

    return model


set_global_seed(42)
resnet_transfer = create_resnet18_transfer(NUM_CLASSES, freeze_backbone=True).to(device)

# Count trainable vs total parameters
total_params = sum(p.numel() for p in resnet_transfer.parameters())
trainable_params = sum(p.numel() for p in resnet_transfer.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")
print(f"Trainable fraction:   {trainable_params/total_params:.2%}")

In [ ]:
# Train transfer learning model (only head)
set_global_seed(42)
resnet_transfer = create_resnet18_transfer(NUM_CLASSES, freeze_backbone=True).to(device)

print("=" * 60)
print("Training ResNet18 Transfer Learning (frozen backbone)")
print("=" * 60)

history_transfer = train_classifier(
    resnet_transfer, train_loader_224, val_loader_224,
    epochs=15, lr=1e-3, patience=5
)

In [ ]:
# Plot transfer learning training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history_transfer["train_loss"]) + 1)
ax1.plot(epochs, history_transfer["train_loss"], label="Train Loss", marker="o", markersize=3)
ax1.plot(epochs, history_transfer["val_loss"], label="Val Loss", marker="s", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Transfer Learning: Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history_transfer["train_acc"], label="Train Acc", marker="o", markersize=3)
ax2.plot(epochs, history_transfer["val_acc"], label="Val Acc", marker="s", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Transfer Learning: Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7. Fine-Tuning

**Strategy:** Start from the transfer-learned model, unfreeze the last few layers of the
backbone, and train with a **much lower learning rate** to avoid destroying learned features.

We use discriminative learning rates:
- Backbone layers: `1e-5`
- New head: `1e-4`

In [ ]:
def unfreeze_resnet_layers(model, unfreeze_from="layer3"):
    """Unfreeze ResNet layers from a given layer onwards.

    ResNet18 layers: conv1, bn1, layer1, layer2, layer3, layer4, fc
    """
    unfreeze = False
    for name, param in model.named_parameters():
        if unfreeze_from in name:
            unfreeze = True
        if unfreeze or "fc" in name:
            param.requires_grad = True


# Start from a fresh transfer model and unfreeze last layers
set_global_seed(42)
resnet_finetune = create_resnet18_transfer(NUM_CLASSES, freeze_backbone=True).to(device)

# Load the best transfer learning weights as starting point
# (In practice you'd save/load, here we re-use the trained model)
resnet_finetune.load_state_dict(resnet_transfer.state_dict())

# Unfreeze layer3, layer4, and fc
unfreeze_resnet_layers(resnet_finetune, unfreeze_from="layer3")

trainable_params = sum(p.numel() for p in resnet_finetune.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet_finetune.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({trainable_params/total_params:.1%})")

# Show which layers are unfrozen
print("\nUnfrozen layers:")
for name, param in resnet_finetune.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

In [ ]:
# Fine-tune with discriminative learning rates
def train_finetune(model, train_loader, val_loader, epochs=10,
                   backbone_lr=1e-5, head_lr=1e-4, patience=5, device=device):
    """Fine-tune with different LRs for backbone and head."""

    # Separate parameter groups
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            if "fc" in name:
                head_params.append(param)
            else:
                backbone_params.append(param)

    optimizer = torch.optim.Adam([
        {"params": backbone_params, "lr": backbone_lr},
        {"params": head_params, "lr": head_lr},
    ], weight_decay=1e-4)

    loss_fn = nn.CrossEntropyLoss()
    early_stop = EarlyStopping(patience=patience)

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": []
    }

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            correct += (output.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # Validate
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = loss_fn(output, y_batch)
                running_loss += loss.item() * X_batch.size(0)
                correct += (output.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if early_stop(val_loss, model):
            print(f"Early stopping at epoch {epoch}")
            early_stop.load_best(model)
            break

    elapsed = time.time() - start_time
    history["training_time"] = elapsed
    print(f"\nFine-tuning completed in {elapsed:.1f}s")
    return history


print("=" * 60)
print("Fine-Tuning ResNet18 (unfrozen layer3+layer4+fc)")
print("=" * 60)

history_finetune = train_finetune(
    resnet_finetune, train_loader_224, val_loader_224,
    epochs=10, backbone_lr=1e-5, head_lr=1e-4, patience=5
)

In [ ]:
# Plot fine-tuning training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history_finetune["train_loss"]) + 1)
ax1.plot(epochs, history_finetune["train_loss"], label="Train Loss", marker="o", markersize=3)
ax1.plot(epochs, history_finetune["val_loss"], label="Val Loss", marker="s", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Fine-Tuning: Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history_finetune["train_acc"], label="Train Acc", marker="o", markersize=3)
ax2.plot(epochs, history_finetune["val_acc"], label="Val Acc", marker="s", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Fine-Tuning: Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 8. Evaluation

We evaluate all three models on the **test set** with:
- Overall accuracy
- Confusion matrix
- Per-class precision, recall, F1-score

In [ ]:
@torch.no_grad()
def get_all_predictions(model, loader, device=device):
    """Collect all predictions and labels from a DataLoader."""
    model.eval()
    all_preds = []
    all_labels = []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        output = model(X_batch)
        preds = output.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())
    return np.array(all_labels), np.array(all_preds)


# Get predictions for all models
print("Evaluating on test set...")

y_true_32, y_pred_baseline = get_all_predictions(baseline_cnn, test_loader_32)
y_true_224, y_pred_transfer = get_all_predictions(resnet_transfer, test_loader_224)
_, y_pred_finetune = get_all_predictions(resnet_finetune, test_loader_224)

acc_baseline = (y_pred_baseline == y_true_32).mean()
acc_transfer = (y_pred_transfer == y_true_224).mean()
acc_finetune = (y_pred_finetune == y_true_224).mean()

print(f"\nTest Accuracy:")
print(f"  Baseline CNN:      {acc_baseline:.4f}")
print(f"  Transfer Learning: {acc_transfer:.4f}")
print(f"  Fine-Tuning:       {acc_finetune:.4f}")

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (y_pred, title) in zip(axes, [
    (y_pred_baseline, f"Baseline CNN (Acc: {acc_baseline:.3f})"),
    (y_pred_transfer, f"Transfer Learning (Acc: {acc_transfer:.3f})"),
    (y_pred_finetune, f"Fine-Tuning (Acc: {acc_finetune:.3f})"),
]):
    y_true_for_cm = y_true_32 if "Baseline" in title else y_true_224
    cm = confusion_matrix(y_true_for_cm, y_pred)
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    # Annotate
    thresh = cm.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontsize=7)

plt.suptitle("Confusion Matrices: All Approaches", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class classification reports
print("=" * 60)
print("BASELINE CNN -- Classification Report")
print("=" * 60)
print(classification_report(y_true_32, y_pred_baseline, target_names=CLASS_NAMES))

print("=" * 60)
print("TRANSFER LEARNING -- Classification Report")
print("=" * 60)
print(classification_report(y_true_224, y_pred_transfer, target_names=CLASS_NAMES))

print("=" * 60)
print("FINE-TUNING -- Classification Report")
print("=" * 60)
print(classification_report(y_true_224, y_pred_finetune, target_names=CLASS_NAMES))

---

## 9. Error Analysis

Analyzing misclassified examples helps us understand failure modes:
- Which classes are most often confused?
- Are errors visually understandable?

In [ ]:
def show_misclassified(model, loader, y_true, y_pred, class_names,
                       n_examples=15, title="Misclassified Examples",
                       mean=None, std=None):
    """Display misclassified images with true and predicted labels."""
    misclassified_idx = np.where(y_true != y_pred)[0]
    if len(misclassified_idx) == 0:
        print("No misclassified examples!")
        return

    # Select random misclassified examples
    np.random.seed(42)
    selected = np.random.choice(
        misclassified_idx, min(n_examples, len(misclassified_idx)), replace=False
    )

    ncols = 5
    nrows = (len(selected) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    if nrows == 1:
        axes = axes.reshape(1, -1)

    # Collect images from the loader's dataset
    for idx_pos, data_idx in enumerate(selected):
        row, col = idx_pos // ncols, idx_pos % ncols
        ax = axes[row, col]

        img, _ = loader.dataset[data_idx]
        if isinstance(img, torch.Tensor):
            img_np = img.numpy().transpose(1, 2, 0)
        else:
            img_np = np.array(img)

        # Denormalize
        if mean is not None and std is not None:
            img_np = img_np * np.array(std) + np.array(mean)
        img_np = np.clip(img_np, 0, 1)

        ax.imshow(img_np)
        true_label = class_names[y_true[data_idx]]
        pred_label = class_names[y_pred[data_idx]]
        color = "red"
        ax.set_title(f"T: {true_label}\nP: {pred_label}", fontsize=9, color=color)
        ax.axis("off")

    # Hide unused axes
    for idx_pos in range(len(selected), nrows * ncols):
        row, col = idx_pos // ncols, idx_pos % ncols
        axes[row, col].axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


# Show misclassified examples for the best model (fine-tuned)
show_misclassified(
    resnet_finetune, test_loader_224, y_true_224, y_pred_finetune,
    CLASS_NAMES, n_examples=15,
    title="Fine-Tuned ResNet18: Misclassified Examples",
    mean=IMAGENET_MEAN, std=IMAGENET_STD
)

In [ ]:
# Most confused class pairs for the fine-tuned model
cm = confusion_matrix(y_true_224, y_pred_finetune)
np.fill_diagonal(cm, 0)  # Zero out diagonal (correct predictions)

# Find top-5 most confused pairs
print("Top-5 Most Confused Class Pairs (Fine-Tuned Model):")
print("-" * 50)
confused_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j:
            confused_pairs.append((cm[i, j], CLASS_NAMES[i], CLASS_NAMES[j]))

confused_pairs.sort(reverse=True)
for count, true_cls, pred_cls in confused_pairs[:5]:
    print(f"  {true_cls:12s} -> {pred_cls:12s}: {count} errors")

---

## 10. Model Saving and Inference

In [ ]:
# Save the best model
best_model_path = os.path.join(SAVE_DIR, "best_image_classifier.pth")

# Determine best model
best_acc = max(acc_baseline, acc_transfer, acc_finetune)
if best_acc == acc_finetune:
    best_model = resnet_finetune
    best_name = "Fine-Tuned ResNet18"
elif best_acc == acc_transfer:
    best_model = resnet_transfer
    best_name = "Transfer Learning ResNet18"
else:
    best_model = baseline_cnn
    best_name = "Baseline CNN"

# Save state dict and metadata
checkpoint = {
    "model_state_dict": best_model.state_dict(),
    "model_name": best_name,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "accuracy": best_acc,
}
torch.save(checkpoint, best_model_path)
print(f"Best model ({best_name}) saved to: {best_model_path}")
print(f"File size: {os.path.getsize(best_model_path) / 1e6:.1f} MB")

In [ ]:
# Inference function
def predict_image(model, image, transform, class_names, device=device):
    """Predict the class of a single image.

    Args:
        model: Trained model.
        image: PIL Image or tensor.
        transform: Preprocessing transform.
        class_names: List of class name strings.

    Returns:
        predicted_class (str), confidence (float), all_probs (dict)
    """
    model.eval()
    if isinstance(image, torch.Tensor):
        from torchvision.transforms.functional import to_pil_image
        image = to_pil_image(image)

    img_t = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_t)
        probs = F.softmax(output, dim=1).cpu().numpy()[0]

    pred_idx = probs.argmax()
    pred_class = class_names[pred_idx]
    confidence = probs[pred_idx]
    all_probs = {name: float(p) for name, p in zip(class_names, probs)}

    return pred_class, confidence, all_probs


# Demo: predict on a few test images
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

for i, ax in enumerate(axes):
    img, true_label = test_dataset_raw[i * 100]
    if isinstance(true_label, torch.Tensor):
        true_label = true_label.item()

    # Choose transform based on best model type
    if "ResNet" in best_name:
        tfm = eval_transform_224
    else:
        tfm = eval_transform_32

    pred_class, confidence, _ = predict_image(best_model, img, tfm, CLASS_NAMES)

    # Display
    if isinstance(img, torch.Tensor):
        img_np = img.numpy().transpose(1, 2, 0)
    else:
        img_np = np.array(img) / 255.0
    img_np = np.clip(img_np, 0, 1)

    ax.imshow(img_np)
    color = "green" if pred_class == CLASS_NAMES[true_label] else "red"
    ax.set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {pred_class} ({confidence:.1%})",
                 fontsize=9, color=color)
    ax.axis("off")

plt.suptitle(f"Inference Demo ({best_name})", fontsize=13)
plt.tight_layout()
plt.show()

---

## 11. Results Comparison

Comprehensive comparison of all three approaches.

In [ ]:
# Comparison table
n_params_baseline = sum(p.numel() for p in baseline_cnn.parameters())
n_params_transfer = sum(p.numel() for p in resnet_transfer.parameters() if p.requires_grad)
n_params_finetune = sum(p.numel() for p in resnet_finetune.parameters() if p.requires_grad)
n_params_total_resnet = sum(p.numel() for p in resnet_transfer.parameters())

print("\n" + "=" * 85)
print(f"{'Approach':<25} {'Test Acc':>10} {'Train Time':>12} {'Trainable Params':>18} {'Total Params':>14}")
print("=" * 85)

rows = [
    ("Baseline CNN", acc_baseline, history_baseline.get("training_time", 0),
     n_params_baseline, n_params_baseline),
    ("Transfer Learning", acc_transfer, history_transfer.get("training_time", 0),
     n_params_transfer, n_params_total_resnet),
    ("Fine-Tuning", acc_finetune, history_finetune.get("training_time", 0),
     n_params_finetune, n_params_total_resnet),
]

for name, acc, t_time, trainable, total in rows:
    print(f"{name:<25} {acc:>10.4f} {t_time:>10.1f}s {trainable:>18,} {total:>14,}")

print("=" * 85)

In [ ]:
# Combined training curves plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

all_histories = {
    "Baseline CNN": history_baseline,
    "Transfer Learning": history_transfer,
    "Fine-Tuning": history_finetune,
}
colors = ["tab:blue", "tab:orange", "tab:green"]

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist["val_loss"]) + 1)
    ax1.plot(epochs, hist["val_loss"], label=name, color=color, linewidth=2)
    ax2.plot(epochs, hist["val_acc"], label=name, color=color, linewidth=2)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation Loss")
ax1.set_title("Validation Loss Comparison")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Validation Accuracy Comparison")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 12. Conclusions and Next Steps

### Key Takeaways

- **Transfer learning** significantly outperforms training from scratch, even on a well-known benchmark like CIFAR-10
- **Fine-tuning** with discriminative learning rates (lower LR for pretrained layers, higher for new head) provides the best results
- The **baseline CNN** has far fewer parameters but cannot match the representational power of a pretrained ResNet18
- **Data augmentation** is critical for the from-scratch CNN to avoid overfitting
- **Early stopping** prevents overfitting and reduces training time

### Common Pitfalls

| Pitfall | Solution |
|---------|----------|
| Forgetting to freeze backbone | Explicitly set `requires_grad = False` |
| Using same LR for all layers in fine-tuning | Use parameter groups with discriminative LRs |
| Augmenting validation/test data | Only augment training data |
| Wrong normalization for pretrained models | Use ImageNet stats for ImageNet-pretrained models |
| Not resizing images for ResNet | ResNet expects 224x224 input |

### Next Steps

- Try other pretrained architectures (EfficientNet, ViT)
- Experiment with learning rate schedules (cosine annealing, one-cycle)
- Apply to a custom dataset with your own images
- Add test-time augmentation for better accuracy
- Deploy the model as an API with Flask/FastAPI